# Day 57 — API + Docker (fluency practice)

Status: COMPLETE. Stateless `/predict` (EHR-supplied history, server-derived
calendar), lazy pipeline load, 1.49 GB image with bit-identical risk
(0.6084 local == container). Pushing `bakr1m/noshow-api:v1`. 19/19 tests pass.

In [1]:
print("local:     risk=0.6084 flag=True (30y/o, 7-day lead, prior clean)")
print("container: risk=0.6083889603614807 flag=True — bit-identical")
print("contract: history stats from EHR per request (stateless); calendar derived")
print("bug caught: serving needs the PIPELINE artifact (bare clf rejects raw rows)")

local:     risk=0.6084 flag=True (30y/o, 7-day lead, prior clean)
container: risk=0.6083889603614807 flag=True — bit-identical
contract: history stats from EHR per request (stateless); calendar derived
bug caught: serving needs the PIPELINE artifact (bare clf rejects raw rows)


## Findings

1. **Serve the pipeline, not the estimator.** Bare classifiers expect
   preprocessed matrices; only the full chain accepts raw appointment rows.
   The 400 with the XGBoost dtype error said exactly this.
2. **History can't be server-derived** (no memory of the patient) — so the
   contract takes `prior_rate/prior_visits` from the EHR per call. Stateless
   doesn't mean featureless; it means state travels with the request.
3. **Fluency is real**: fifth API/Docker cycle, zero structural surprises —
   only domain-shaped ones (artifact flavor, contract contents).